# 1.2 A2/A3/A5 芯片架构简介

本节介绍昇腾 AI 处理器 A2、A3、A5 系列芯片的AI CORE架构。

---

## 学习目标

完成本节后，你将能够：
- 理解昇腾 AI 处理器的整体架构
- 掌握 AI Core 的 Vector 计算单元结构
- 了解不同系列芯片的架构差异

---

## 1. 昇腾 AI 处理器架构概述

昇腾 AI 处理器采用达芬奇架构，专为 AI 计算优化设计。整体架构包括：
- **AI Core**：核心计算单元，包含 Vector、Cube、Scalar 等计算模块
- **Buffer**：片上缓存，包括 UB（Unified Buffer）、L1/L2 缓存
- **DMA**：数据搬运引擎

### 1.1 AI Core 内部结构

AI Core 是昇腾处理器的核心计算单元，包含：
```text
┌─────────────────────────────────────┐
│              AI Core                │
│                                     │
│    ┌───────────┐     ┌───────────┐  │
│    │  Buffer   │     │  Vector   │  │
│    │ (UB/L1/L2)│     │   Unit    │  │
│    └───────────┘     └───────────┘  │
│         ↑            ┌───────────┐  │
│         │            │  Scalar   │  │
│         │            │   Unit    │  │
│         │            └───────────┘  │
│    ┌───────────┐     ┌───────────┐  │
│    │    DMA    │     │  Cube     │  │
│    │    Unit   │     │  Unit     │  │
│    └───────────┘     └───────────┘  │
│          │                          │
│          │                          │
└──────────↓──────────────────────────┘
    ┌─────────────┐                    
    │    HBM      │                    
    │             │                    
    └─────────────┘           
```         
#### Vector 计算单元

- 负责向量计算，如逐元素算术运算、类型转换等
- 支持 SIMD（单指令多数据）/SIMT（单指令多线程，仅A5支持）并行计算
- 每次可处理多个数据元素

#### Cube 计算单元

- 专门用于矩阵乘法运算
- 支持高性能矩阵乘累加运算

#### Scalar 计算单元

- 处理标量运算和控制逻辑
- 支持分支、循环等控制流

#### DMA 搬运单元

- 数据搬入单元MTE2，将Global Memory数据搬运到Local Memory
- 数据搬出单元MTE3，将Local Memory数据搬运到Global Memory

---

## 2. Vector 计算单元详解

### 2.1 Vector Unit 功能

Vector 计算单元支持以下操作：
- **算术运算**：加减乘除、绝对值、平方等
- **逻辑运算**：与、或、非、异或等
- **比较运算**：大于、小于、等于等
- **数学函数**：sin、cos、exp、log等
- **类型转换**：FP16↔FP32、INT8↔FP32等

### 2.2 SIMD 并行度

Vector Unit 采用 SIMD 架构，单条指令可同时处理多个数据：
- FP32：通常一次处理64个元素
- FP16：通常一次处理128个元素
- INT8：通常一次处理256个元素

---

## 3. 存储层次结构
```text
Global Memory (HBM)
        ↑
        │ DMA
        ↓
    L2 Cache
        ↑
        │ DMA
        ↓
    L1 Cache
        ↑
        │
        ↓
    Unified Buffer (UB)
        ↑
        │
        ↓
    Vector Registers
```
### 3.1 Unified Buffer (UB)

- Vector 计算的主要工作区
- 容量：A2/A3为192KB, A5为256KB（实际可用 248KB，框架预留 8KB）
- 支持 DataCopy 快速搬运
- 需要合理规划以最大化性能

### 3.2 L2 Cache

- HBM和AI Core UB之间的缓存，数据存在于L2 Cache中时AI CORE会优先从L2 Cache中读取。
- 常用数据优先存储在L2 Cache中以提升访问效率

---

## 4. 编程模型

### 4.1 MemBase编程

- 单指令控制多个数据
- 多个 AI Core 并行工作
- 通过 Tiling 机制进行数据分块
- 通过指令直接操作在UB中的数据

### 4.2 RegBase 编程

- 仅A5芯片支持
- 单指令处理256个字节数据
- 多个 AI Core 并行工作
- 通过 Tiling 机制进行数据分块
- 将数据从UB加载到Register(寄存器)中，通过微指令处理多个数据

### 4.3 SIMT 编程

- 仅A5芯片支持
- 单指令多线程处理多个数据
- 多个 AI Core 并行工作
- 通过线程排布进行数据分块

---

## 小结

本节介绍了昇腾 AI 处理器的架构，特别是 Vector 计算单元的结构和功能。关键要点：

1. AI Core 包含 Vector、Cube、Scalar 三种计算单元以及DMA搬运单元
2. Vector Unit 支持 SIMD/SIMT(A5) 并行计算
3. 不同系列芯片在计算能力和缓存容量上有差异
4. 存储层次：HBM → L2 → L1 → UB → Registers
5. UB 是 Vector 计算的主要工作区，A2/A3容量为192KB，A5为256KB（实际可用 248KB，框架预留 8KB）

## 课后练习

1. 简述昇腾 AI 处理器的整体架构
2. Vector Unit 主要支持哪些类型的运算？

**执行以下代码获取答案。**

上一节：[1.1 章节介绍](./01.01_chapter_intro.ipynb)</cell_id>


In [ ]:
!cat ./answer/01.02_answer.txt